In [ ]:
from langchain_community.document_loaders import PyPDFLoader, TextLoader

In [2]:
file_path =  "../data/2404.12195v1.pdf"
loader = PyPDFLoader(file_path, mode="single")
loaded_data = loader.load()

# Text Splitting

In [3]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [4]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100)

In [5]:
splitted_pdf = text_splitter.split_documents(loaded_data)

In [6]:
splitted_pdf

[Document(metadata={'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2024-04-19T00:40:42+00:00', 'author': '', 'keywords': '', 'moddate': '2024-04-19T00:40:42+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'subject': '', 'title': '', 'trapped': '/False', 'source': '../data/2404.12195v1.pdf', 'total_pages': 26}, page_content='OPEN BEZOAR : S MALL , COST-EFFECTIVE AND OPEN MODELS\nTRAINED ON MIXES OF INSTRUCTION DATA\nChandeepa Dissanayake, Lahiru Lowe, Sachith Gunasekara, and Yasiru Ratnayake\nSurge Global\n{chandeepa, lahiru.lowe, sachith, yasiru}@surge.global\nApril 19, 2024\nABSTRACT\nInstruction fine-tuning pretrained LLMs for diverse downstream tasks has demonstrated remarkable\nsuccess and has captured the interest of both academics and practitioners. To ensure such fine-tuned\nLLMs align with human preferences, techniques such as RLHF and DPO have emerged. At the\nsame time, ther

# Vector Store

In [9]:
import os
from langchain_pinecone import PineconeVectorStore 
from pinecone import Pinecone
from langchain_openai import OpenAIEmbeddings

In [11]:
pc_key = Pinecone(os.getenv("PINECONE_API_KEY"))
pc_index = pc_key.Index("assignment-index-stemlink")

In [12]:
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
vector_store = PineconeVectorStore(embedding=embeddings,index=pc_index)

In [14]:
from uuid import uuid4

document_ids = [str(uuid4()) for _ in range(len(splitted_pdf))]

In [15]:
vector_store.add_documents(documents=splitted_pdf,ids=document_ids)

['34e26b13-abe0-45e4-93c7-92396a112cb1',
 '3cc19e11-d7e8-4ab9-acb1-32cbd8781510',
 'a0315919-5f11-4c77-b3dd-fb02e6e0a07a',
 '4cf2f46c-e097-45ed-a371-3fe679e9563a',
 'd75c4b77-dae6-4f0d-92f5-290164457a40',
 '0cfb6cc2-5495-4cd3-9b71-d22de12da9ee',
 '54f8464a-fa74-4f9a-b756-55d004cdcb7a',
 'dc6276c3-bc2b-4d5d-91c5-12edbe4116e8',
 '1f6269bc-4ec2-473b-8ef3-1eeff6c5d00e',
 '8e94c0d1-bc1d-468b-baf0-9b33271efe1b',
 '9e4d71bb-f4a5-463f-9c30-841307706b7c',
 'df1e6876-9f39-4989-93f2-6ec10fb68e6c',
 '3b184574-caf9-44f7-aec3-7ae78ad50118',
 'd195b70a-2981-4af3-b4d6-3f027adc5ee3',
 'a80d8fef-779b-4a8a-8219-c9904963837c',
 '48decc30-bed1-4e25-8885-10225d00a143',
 'cfd0338b-7750-47f0-acaf-aab5d3ff26b6',
 '4326586a-f8d6-47d8-a2bb-d510086f9261',
 'b2fdbf3e-dce8-4f52-b2bd-cf95f04646ec',
 'e18db9fd-6323-4eb2-b9fd-5028a600628f',
 '4b1f3a0b-fb61-41a1-b702-026977c1ac00',
 'f3ec2de6-091d-4144-bec9-e2de13940025',
 '20cd66a2-05ab-41d6-8278-9a57e95900c1',
 '784b480f-2c63-4841-b3fd-41e9c9dc267f',
 '1f191187-31e1-

# Retrievel

In [21]:
query = "How many datasets were generated by the authors? What are they?"
results = vector_store.similarity_search_with_score(query, k=1)

In [27]:
results[0][0].page_content

'their respective dataset generation. However, to promote open source practices, we selected models without restrictions\non commercial use of their generated content. Through our exploration, we identified several suitable models and\nultimately chose h2oai/h2ogpt-gm-oasst1-en-2048-falcon-40b-v2 [10].\nIn both the LaMini and Evol-Instruct methods, we utilized the databricks/databricks-dolly-15k dataset [5] to\nselect seed instructions as examples for the new dataset. This dataset contains instruction/response pairs suitable for\ninstruction-tuning pretrained models, dispersed among the following categories: Creative Writing, Closed Question\nAnswering (QA), Open QA, Summarization, Information Extraction, Classification, and Brainstorming.\nFor the Orca scheme, we used the FLAN-v2 Collection [28] to select query and response pairs, following the\nmethodology of the Orca paper authors [22]. The FLAN-v2 dataset comprises several submixtures, including Flan2021'